# MediSight AI — Clinical Risk Prediction & Explainable AI

This notebook documents the current ML pipeline used by MediSight AI. It uses Synthea-generated synthetic healthcare records to build a forward-looking clinical risk model and explain predictions with SHAP.

> **Important:** Synthea data is synthetic. These results demonstrate the MVP ML pipeline and do not represent clinical performance or medical advice.

## 1. ML objective

Predict whether a patient will have a future inpatient or emergency encounter using information available up to a patient-specific cutoff date. The model is intended as **decision support**, not diagnosis.

## 2. Dataset

The current MVP uses **30,921 Synthea-generated synthetic patient records** from two populations. The feature table was created by `ml/src/build_synthea_features.py` from patient, encounter, condition, medication, and relevant observation/lab data.

The target is a forward-looking inpatient/emergency encounter label. Because the source is synthetic, the model metrics should be interpreted as engineering/ML validation results rather than clinical evidence.

In [6]:
from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent.parent

DATA_PATH = ROOT / 'ml' / 'data' / 'synthea_features.csv'
MODEL_PATH = ROOT / 'ml' / 'models' / 'risk_model.joblib'
META_PATH = ROOT / 'ml' / 'models' / 'model_metadata.json'

df = pd.read_csv(DATA_PATH)
metadata = json.loads(META_PATH.read_text(encoding='utf-8'))

print('Dataset shape:', df.shape)
print('Model:', MODEL_PATH)
print('Metadata loaded:', bool(metadata))

Dataset shape: (30921, 14)
Model: c:\Users\ASUS\Downloads\medisight-ai-updated\medisight-ai\ml\models\risk_model.joblib
Metadata loaded: True


In [7]:
df.head()

,PATIENT,age,is_male,num_active_conditions,num_visits_last_year,num_prescriptions,abnormal_lab_ratio,has_diabetes,has_hypertension,has_heart_disease,latest_glucose,latest_systolic_bp,label,population
0,5605b66b-e92d-c16c-1b83-b8bf7040d51f,41.478439,0.0,9.0,2,4.0,0.875000,0.0,1.0,0.0,90.0,147.0,1,pop1
1,c06a2ab3-c3b1-a3d0-7c12-262a12c6885e,75.397673,1.0,8.0,2,4.0,0.000000,0.0,0.0,1.0,90.0,120.0,0,pop1
2,6e5ae27c-8038-7988-e2c0-25a103f01bfa,68.936345,1.0,6.0,1,2.0,0.000000,0.0,1.0,0.0,90.0,120.0,1,pop1
3,94d39619-8e17-d248-41b3-9ec1f2d031b2,53.221081,1.0,9.0,14,2.0,0.301887,0.0,0.0,0.0,89.7,118.0,1,pop1
4,23832f5d-e045-2541-1626-a65dce9bbcf7,42.529774,0.0,6.0,1,4.0,0.142857,0.0,0.0,1.0,90.0,117.0,0,pop1


In [8]:
# Identify the target column and summarize the class balance.
target_candidates = [c for c in df.columns if c.lower() in {'target', 'label', 'risk_label', 'future_event', 'future_inpatient_or_emergency'}]
print('Columns:', df.columns.tolist())
print('Candidate target columns:', target_candidates)

Columns: ['PATIENT', 'age', 'is_male', 'num_active_conditions', 'num_visits_last_year', 'num_prescriptions', 'abnormal_lab_ratio', 'has_diabetes', 'has_hypertension', 'has_heart_disease', 'latest_glucose', 'latest_systolic_bp', 'label', 'population']
Candidate target columns: ['label']


## 3. Current training result

The production model was trained with `ml/src/train_model_synthea.py` on the current Synthea feature table. The held-out test results are:

| Metric | Result |
|---|---:|
| Accuracy | 67.68% |
| Precision | 79.21% |
| Recall | 68.46% |
| F1 | 73.44% |
| ROC-AUC | 73.89% |
| Test patients | 7,731 |

These are held-out **synthetic-data** results.

In [9]:
# Load the already-trained production model used by the backend.
model = joblib.load(MODEL_PATH)
print(type(model).__name__)
print('Model loaded successfully.')

XGBClassifier
Model loaded successfully.


## 4. Explainable AI with SHAP

SHAP is used to explain how input features contribute to an individual model prediction and to inspect global feature influence. The explanation describes the model's behavior; it is not a medical diagnosis.

In [10]:
import shap

print('SHAP version:', shap.__version__)
print('SHAP import successful.')

SHAP version: 0.52.0
SHAP import successful.


## 5. Reproducible evaluation note

For a full retraining/evaluation run, execute from the project root:

```text
python ml\\src\\build_synthea_features.py
python ml\\src\\train_model_synthea.py
```

The notebook is intentionally focused on documenting and explaining the current trained pipeline rather than generating a second, inconsistent synthetic dataset.

## 6. Limitations

- Synthea records are synthetic and do not establish clinical validity.
- The model should be treated as decision support, not diagnosis.
- Real-world deployment would require external clinical validation, monitoring, calibration, privacy review, and appropriate governance.
- Model performance can change when the data distribution changes.